# 05 · Task 5 — own idea, and the report

*Compressed spectral loss + the findings of notebooks 02–04*

Neural Audio and Speech Processing — Day 2 / Application to Speech Enhancement
Homework assignment, R. Scheibler (2026-07-02) · dataset: Voicebank-DEMAND (16 kHz)

> **Task 5.** … (your own idea.)
> **Deliverable.** `python ./create_report.py runs/<baseline> runs/<myexperiment>`
> plus a section describing the experiment.

## The idea: fix the loss, then stack the wins

The baseline optimises

```
loss = 1.0 · MSE(|Ŝ|, |S|) + 0.01 · (−SI-SDR)
```

Two things are wrong with the first term for speech:

1. **It is dominated by loud bins.** Magnitude spectrograms have ~60 dB of dynamic
   range, so an MSE on linear magnitudes spends almost all of its gradient on a few
   high-energy low-frequency bins and effectively ignores everything that carries
   intelligibility. The standard fix in the speech-enhancement literature is a
   **power-law compression**: compare `|Ŝ|^c` to `|S|^c` with `c ≈ 0.3`, which is a
   rough loudness law and equalises the contribution across bins.
2. **The two terms are wildly out of scale.** The MSE is ~1e-2 while `−SI-SDR` is
   ~−10, so with weight 0.01 both contribute comparably — but compressing the
   magnitudes changes the MSE scale, so the weights are re-tuned together.

`models/crn.py` therefore also takes `compression`, `spec_weight` and `sdr_weight`,
with defaults that reproduce the baseline exactly:

```python
def compressed_mse(pred_mag, target_mag, compression=1.0, eps=1e-6):
    if compression != 1.0:
        pred_mag = pred_mag.clamp_min(eps).pow(compression)
        target_mag = target_mag.clamp_min(eps).pow(compression)
    return F.mse_loss(pred_mag, target_mag)
```

The final experiment combines this with the best activation (notebook 02), the best
learning rate (notebook 03) and the shortened warmup (notebook 04).

In [ ]:
# --- Google Colab bootstrap (does nothing when running locally) --------------
# IMPORTANT: point this at the fork that contains the homework modifications
# (models/crn.py with a selectable activation, train.py with --warmup-steps).
REPO_URL = "https://github.com/Ahmed-AlGhosaini/nanoSE.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os

    if not os.path.exists("nanoSE"):
        !git clone -q $REPO_URL nanoSE
    %cd nanoSE
    !pip install -q -r requirements.txt

    import torch
    if not torch.cuda.is_available():
        print("No GPU! Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
import sys
from pathlib import Path

# Make notebooks/nb_utils.py importable no matter where the kernel was started
for candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (candidate / "nb_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import pandas as pd

import nb_utils

ROOT = nb_utils.bootstrap()   # chdir to the repository root + print the device

In [ ]:
SWEEP_EPOCHS = 5
FULL_EPOCHS = 25

registry = nb_utils.registry()
best_activation = registry.get("best_activation_name", "leaky_relu")
best_lr = float(registry.get("best_lr_value", 1e-3))
print(f"carried over: activation={best_activation}, lr={best_lr:g}")

## Ablation — is the compressed loss actually the ingredient that helps?

Four short runs isolate the loss change from the hyperparameters, so the report can
attribute the gain instead of just claiming it.

> **Resuming after a disconnect.** Every finished run is recorded in
> `notebooks/experiment_runs.json`, and the sweep loops skip anything already recorded.
> Re-running the cell after a Colab timeout continues where it stopped instead of
> starting over.

In [ ]:
ablation_variants = {
    "baseline loss":            dict(compression=1.0, spec_weight=1.0, sdr_weight=0.01),
    "compressed (c=0.3)":       dict(compression=0.3, spec_weight=1.0, sdr_weight=0.01),
    "compressed + sdr x10":     dict(compression=0.3, spec_weight=1.0, sdr_weight=0.1),
    "compressed (c=0.5)":       dict(compression=0.5, spec_weight=1.0, sdr_weight=0.1),
}

ablation_runs = {}
for label, loss_kwargs in ablation_variants.items():
    slug = (label.replace(" ", "_").replace("(", "").replace(")", "")
                 .replace("=", "").replace(".", "").replace("+", ""))
    key = f"loss_{slug}"
    done = nb_utils.recall(key)
    if done is not None:
        ablation_runs[label] = done
        print(f"[skip] {label:<24} -> {done}")
        continue

    model_expr = (
        f'CRNTiny(activation="{best_activation}", '
        f'compression={loss_kwargs["compression"]}, '
        f'spec_weight={loss_kwargs["spec_weight"]}, '
        f'sdr_weight={loss_kwargs["sdr_weight"]})'
    )
    config = nb_utils.write_config(
        f"exp_{key}.py",
        name=key,
        model=model_expr,
        docstring=f"Task 5 ablation: {label}.",
        epochs=SWEEP_EPOCHS,
        lr=best_lr,
    )
    ablation_runs[label] = nb_utils.remember(key, nb_utils.run_training(config))

In [ ]:
labels = list(ablation_runs)
runs = [ablation_runs[label] for label in labels]

table = nb_utils.summarize(runs, labels=labels, best_epoch=True)
display(table[["label", "epoch", "val_si_sdr", "pesq", "estoi", "dnsmos"]].round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
reference = table.set_index("label")
for ax, metric, name in zip(axes, ["val_si_sdr", "pesq", "estoi"], ["SI-SDR (dB)", "PESQ", "eSTOI"]):
    nb_utils.plot_bars(list(table.label), list(table[metric]), title=name, ylabel=name,
                       baseline=float(reference.loc["baseline loss", metric]), ax=ax)
plt.tight_layout()
plt.show()

## The final experiment — `config/my_experiment.py`

The winning loss setting is combined with the best activation, learning rate and
warmup, and trained at full length. This is the `runs/<myexperiment>` of the
deliverable.

In [ ]:
best_loss_label = table.iloc[0].label
best_loss = ablation_variants[best_loss_label]
print("best loss configuration:", best_loss_label, best_loss)

model_expr = (
    f'CRNTiny(activation="{best_activation}", '
    f'compression={best_loss["compression"]}, '
    f'spec_weight={best_loss["spec_weight"]}, '
    f'sdr_weight={best_loss["sdr_weight"]})'
)

my_config = nb_utils.write_config(
    "my_experiment.py",
    name="my_experiment",
    model=model_expr,
    docstring=(
        "Task 5 -- power-law compressed spectral loss, combined with the best\n"
        "activation (notebook 02), learning rate (notebook 03) and warmup (notebook 04)."
    ),
    epochs=FULL_EPOCHS,
    lr=best_lr,
    warmup_steps=100,
)
print(my_config.read_text())

In [ ]:
my_run = nb_utils.recall("my_experiment")
if my_run is None:
    my_run = nb_utils.remember("my_experiment", nb_utils.run_training(my_config))

baseline_run = nb_utils.recall("baseline")
assert baseline_run is not None, "Run 01_baseline.ipynb first -- the report needs both runs."
print("baseline     :", baseline_run)
print("my_experiment:", my_run)

In [ ]:
comparison = [baseline_run, my_run]
comparison_labels = ["baseline", "my_experiment"]

summary = nb_utils.summarize(comparison, labels=comparison_labels)
display(summary.round(3))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, metric in zip(axes.ravel(), ["val_si_sdr", "pesq", "estoi", "dnsmos"]):
    nb_utils.plot_curves(comparison, labels=comparison_labels, metric=metric, ax=ax)
plt.tight_layout()
plt.show()

## Listening test

Numbers are not the whole story: over-suppression sounds bad long before SI-SDR
notices. `enhance.py` runs a checkpoint on any wav file — here on a test utterance.

In [ ]:
import subprocess
import sys

import soundfile as sf
from IPython.display import Audio, display

from dataset import VoiceBankDemandDataset

test_ds = VoiceBankDemandDataset(split="test")
noisy_wav, clean_wav = test_ds[7]

Path("demo").mkdir(exist_ok=True)
sf.write("demo/noisy.wav", noisy_wav.numpy(), 16000)
sf.write("demo/clean.wav", clean_wav.numpy(), 16000)

checkpoints = sorted((Path(my_run) / "checkpoints").glob("*.pt"))
checkpoint = checkpoints[-1]
print("checkpoint:", checkpoint)

subprocess.run(
    [sys.executable, "enhance.py", "demo/noisy.wav", "demo/enhanced.wav",
     "--checkpoint", str(checkpoint), "--config", str(Path(my_run) / "config.py")],
    check=True,
)

for name in ("noisy", "enhanced", "clean"):
    print(name)
    display(Audio(f"demo/{name}.wav"))

In [ ]:
import numpy as np

from model import waveform_to_spectrogram

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ("noisy", "enhanced", "clean")):
    audio, _ = sf.read(f"demo/{name}.wav", dtype="float32")
    spec = torch.abs(waveform_to_spectrogram(torch.from_numpy(audio).unsqueeze(0)))[0]
    ax.imshow(np.log1p(spec.numpy()), aspect="auto", origin="lower", cmap="viridis")
    nb_utils.style_axes(ax, name.capitalize(), "Time frames", "Frequency bins" if ax is axes[0] else "")
    ax.grid(False)
plt.tight_layout()
plt.show()

## The deliverable — `report/nanose.md`

`create_report.py` produces the comparison tables; the cell below prepends the
student information and appends the experiment and results sections, exactly as
the assignment asks.

**Edit `STUDENTS` before running it.**

In [ ]:
STUDENTS = [
    # ("Full name", "student number"),
    ("Ahmed Al-Ghosaini", "TODO-student-number"),
]

In [ ]:
tables = nb_utils.create_report(comparison, output="report/_tables.md", best_epoch=True)
# keep only the tables, the header is rewritten below
tables = tables.split("\n", 2)[2].strip()

baseline_history = nb_utils.load_metrics(baseline_run)
my_history = nb_utils.load_metrics(my_run)
unprocessed = baseline_history[baseline_history.epoch == 0].iloc[0]
baseline_best = baseline_history[baseline_history.epoch > 0].loc[lambda d: d.val_si_sdr.idxmax()]
my_best = my_history[my_history.epoch > 0].loc[lambda d: d.val_si_sdr.idxmax()]

students_md = "\n".join(f"* {name} — {number}" for name, number in STUDENTS)

report = f"""# nanoSE Homework — Speech Enhancement Experiment

{students_md}

Dataset: Voicebank-DEMAND (16 kHz). Model: `CRNTiny` (~175 k parameters).
Baseline: `config/default_config.py`. Experiment: `config/my_experiment.py`.

## 1. Experiment

The baseline trains `CRNTiny` to predict a magnitude mask, optimising
`1.0 · MSE(|Ŝ|,|S|) + 0.01 · (−SI-SDR)`. A linear-magnitude MSE is dominated by the
few high-energy low-frequency bins, so most of the gradient is spent where the
signal is already loud rather than where intelligibility lives.

**Modification.** The spectral term uses power-law compressed magnitudes,
`MSE(|Ŝ|^c, |S|^c)` with `c = {best_loss["compression"]}`, and the SI-SDR term is
re-weighted to `{best_loss["sdr_weight"]}` to keep the two contributions comparable
after the rescaling. This is combined with the settings selected in the earlier
notebooks: `{best_activation}` activations in `EncoderBlock`/`DecoderBlock`,
peak learning rate `{best_lr:g}`, and a 100-step LR warmup instead of 500.

**Methodology.** Every candidate was ranked with a short {SWEEP_EPOCHS}-epoch run at
fixed seed 42 and identical data, then the winner was retrained for the full
{FULL_EPOCHS} epochs so that baseline and experiment differ only in the change under
test. Metrics are computed by `train.py` on the Voicebank-DEMAND test split:
SI-SDR over all utterances, PESQ / eSTOI / DNSMOS over the first 20. Reported
numbers are the best epoch by validation SI-SDR.

## 2. Results

{tables}

Reference point: the unprocessed noisy input scores
{unprocessed.val_si_sdr:.2f} dB SI-SDR / PESQ {unprocessed.pesq:.2f} /
eSTOI {unprocessed.estoi:.2f} / DNSMOS {unprocessed.dnsmos:.2f}.

| | Baseline | Experiment | Δ |
|---|---|---|---|
| Val SI-SDR (dB) | {baseline_best.val_si_sdr:.2f} | {my_best.val_si_sdr:.2f} | {my_best.val_si_sdr - baseline_best.val_si_sdr:+.2f} |
| PESQ | {baseline_best.pesq:.2f} | {my_best.pesq:.2f} | {my_best.pesq - baseline_best.pesq:+.2f} |
| eSTOI | {baseline_best.estoi:.2f} | {my_best.estoi:.2f} | {my_best.estoi - baseline_best.estoi:+.2f} |
| DNSMOS | {baseline_best.dnsmos:.2f} | {my_best.dnsmos:.2f} | {my_best.dnsmos - baseline_best.dnsmos:+.2f} |
| Best epoch | {int(baseline_best.epoch)} | {int(my_best.epoch)} | |

## 3. Discussion

TODO — write two or three sentences about what the numbers show. Points to cover:
whether the compressed loss helps PESQ/eSTOI more than SI-SDR (it optimises
perceptual bins rather than waveform energy), whether the gain survived the move
from the {SWEEP_EPOCHS}-epoch sweep to the {FULL_EPOCHS}-epoch run, and what the
ablation in notebook 05 attributes to the loss change versus the hyperparameters.

## 4. Reproduction

```bash
python ./train.py --config config/default_config.py     # baseline
python ./train.py --config config/my_experiment.py      # experiment
python ./create_report.py --best-epoch --output report/nanose.md \\
    {baseline_run} {my_run}
```
"""

Path("report").mkdir(exist_ok=True)
Path("report/nanose.md").write_text(report, encoding="utf-8")
print(report)

## Deliverable checklist

* [ ] `report/nanose.md` — names and student numbers filled in, discussion written
* [ ] the comparison table from `create_report.py` is in the report
* [ ] a section describing the experiment and the methodology
* [ ] a section describing the result
* [ ] markdown or PDF (`jupyter nbconvert` or any markdown-to-PDF tool)

```bash
# optional: markdown -> PDF
pip install md2pdf && md2pdf report/nanose.md report/nanose.pdf
```